In [1]:
from Parts.imports import *
from Parts.data_loader import DatasetLoader, DatasetLoaderV2
from Parts.models import U_NET_VANILLA, U_NET_RESNET, U_NET_RESNET_ATTENTION, U_NET_PLUS_PLUS, TRANS_U_NET
from Parts.training_loop import TrainingLoop, TrainingLoopAdvanced, SaveState, EarlyStopping
from Parts.losses import DiceLoss, Mixed_Dice_Sigmoid

DEBUG = False

In [2]:
pipe = albumentations.Compose([
    albumentations.OneOf([
        albumentations.ElasticTransform(p=1),
        albumentations.GridDistortion(p=1)
    ]),
    albumentations.CLAHE(),
    albumentations.RandomBrightnessContrast()
])

In [3]:
class_instance : DatasetLoader = DatasetLoaderV2(fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\images", fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\masks", (256, 256), pipe)
dataset_loader = DataLoader(class_instance,
                            batch_size=1)

In [4]:
if DEBUG :
    image, mask = next(iter(dataset_loader))
    plt.imshow(mask[0].permute(1, 2, 0))

In [5]:
model = U_NET_VANILLA()
optm = torch.optim.Adam(model.parameters())
loss = DiceLoss()

In [6]:
# trainer = TrainingLoop(model, loss, optm, 6)

In [7]:
# trainer.train(dataset_loader)

In [8]:
early_stopper = EarlyStopping()
saver = SaveState('SimpleUNet', False)

In [9]:
trainier = TrainingLoopAdvanced(model, loss, optm, 1, early_stopper, saver)

In [10]:
from torch.utils.data import Subset # <----------- Such convinient thing they have
subset_class_instance = Subset(class_instance, [0, 1, 2])
subset_test_data_loader = DataLoader(subset_class_instance, 1)

val_subset = Subset(class_instance, [3])
val_subset_loader = DataLoader(val_subset, 1)

In [11]:
trainier.train(subset_test_data_loader, val_subset_loader)

100%|██████████| 3/3 [00:06<00:00,  2.07s/it]

:: current loss :: 0.1367470622062683 :: current validaton 0.0686941146850586 :: current epoch 0
:: State(s) Saved @ current epoch :: 0 @ current val_loss :: 0.0686941146850586


100%|██████████| 3/3 [00:08<00:00,  2.84s/it]

:: State(s) Saved @ current epoch :: -1 @ current val_loss :: 0.0686941146850586
